# Run all scripts — Paper 3 (Colab)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/songwutsa/paper3-pm25-healthcare-visits-saraburi/blob/main/notebooks/run_all_scripts.ipynb)

This notebook runs the 12 Python scripts in `scripts/` in run order and shows their tables and figures, so the
script steps can be checked in Colab without a terminal. Open it in Colab and choose **Runtime ▸ Run all**
(about 5 minutes; step 09 takes about 3 minutes).

The notebook steps (01, 03, 04, 11, 14, 16) are not run here; open those notebooks separately (see the README).
Steps that need their output (10 needs 09; 12 and 13 need 11) read the stored results in `results/`.

| Step | Script | Manuscript |
|---|---|---|
| 00 | `00_download_hdc_monthly_visits.py` | HDC download — online, off by default |
| 02 | `02_validation_anomaly_r2.py` | Sect. 3.1 anomaly R² |
| 05–07 | `05_fig2_…`, `06_fig3_…`, `07_fig4_…` | Figs 2–4 |
| 08–10 | `08_table3_moran_lisa.py`, `09_table3_spacetime_scan.py`, `10_fig5_scan_cluster_maps.py` | Table 3, Fig. 5 |
| 12–13 | `12_fig6_irr_by_lag.py`, `13_fig8_predicted_risk_maps.py` | Figs 6, 8 |
| 15 | `15_build_extended_panel.py` | extended panel for sensitivity (vii) |
| 17 | `17_sensitivity_vii_station_scale.py` | Sect. 3.7 station-scale ratio |

Outputs are written to `colab_outputs/` in the clone, and each step is compared with the stored version in
`figures/`, `results/` or `data/processed/`.

In [ ]:
# ===== Repository =====
import os, sys, subprocess
REPO_URL = "https://github.com/songwutsa/paper3-pm25-healthcare-visits-saraburi"
REPO_DIR = None
for _d in (os.getcwd(), os.path.dirname(os.getcwd()), "/content/paper3-pm25-healthcare-visits-saraburi"):
    if os.path.isdir(os.path.join(_d, "scripts")) and os.path.isdir(os.path.join(_d, "data", "processed")):
        REPO_DIR = _d; break
if REPO_DIR is None:
    REPO_DIR = "/content/paper3-pm25-healthcare-visits-saraburi"
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
OUT = "colab_outputs"; os.makedirs(OUT, exist_ok=True)
print("Repository:", REPO_DIR)

# packages that Colab does not preinstall
need = [p for p in ("libpysal", "esda", "mapclassify") if subprocess.run([sys.executable, "-c", f"import {p}"], capture_output=True).returncode]
if need: subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
print("installed:", need or "nothing needed")

In [ ]:
# ===== Helpers =====
import time, numpy as np, pandas as pd
from IPython.display import Image, display
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

def run(*args):
    """Run one script with the repository as working directory and print its output."""
    t = time.time()
    r = subprocess.run([sys.executable, *args], capture_output=True, text=True)
    print(r.stdout[-6000:]); 
    if r.returncode: print(r.stderr[-4000:]); raise RuntimeError(f"{args[0]} failed")
    print(f"✓ {args[0]} ({time.time() - t:.0f} s)")

def show(png, width=900):
    display(Image(filename=png, width=width))

def same_csv(new, ref, keys=None, tol=1e-9):
    a = pd.read_csv(new, encoding="utf-8-sig"); b = pd.read_csv(ref, encoding="utf-8-sig")
    if keys: a, b = a.sort_values(keys).reset_index(drop=True), b.sort_values(keys).reset_index(drop=True)
    num = a.select_dtypes("number").columns.intersection(b.columns)
    ok = a.shape == b.shape and all(np.allclose(a[c], b[c], atol=tol, equal_nan=True) for c in num)
    print(("✓ matches " if ok else "⚠ differs from ") + ref)
    return ok

RUN_DOWNLOAD = False   # True = step 00 downloads HDC data again (online, several minutes)

## 00 · HDC download (Sect. 2.2, Table 2)
Off by default: the downloaded table is already in `data/processed/`. Set `RUN_DOWNLOAD = True` above to
download again from MOPH Open Data into `colab_outputs/`. MOPH sites often block foreign IP addresses such as
Colab's; if the download fails, run the script on a computer with a Thai internet connection (options are in the
script's header).

In [ ]:
if RUN_DOWNLOAD:
    run("scripts/00_download_hdc_monthly_visits.py", "--start", "2565", "--out", f"{OUT}/hdc_saraburi_pm25_disease_tambon_monthly.csv")
else:
    print("skipped — using data/processed/hdc_saraburi_pm25_disease_tambon_monthly_2565_2569.csv")

## 02 · Agreement on deseasonalised monthly anomalies (Sect. 3.1)
Expected pooled anomaly R² 0.33.

In [ ]:
run("scripts/02_validation_anomaly_r2.py", "data/processed/sat_validation_monthly_pairs.csv")

## 05 · Fig. 2 — satellite vs station PM2.5

In [ ]:
run("scripts/05_fig2_satellite_validation.py", "data/processed/sat_validation_monthly_pairs.csv", f"{OUT}/Figure_2")
show(f"{OUT}/Figure_2.png")

## 06 · Fig. 3 — seasonal and monthly satellite PM2.5

In [ ]:
run("scripts/06_fig3_pm25_seasonal_monthly.py", "data/processed/sat_pm25_tambon_monthly.csv", f"{OUT}/Figure_3")
show(f"{OUT}/Figure_3.png")

## 07 · Fig. 4 — mean PM2.5, downwind indicator and roadside population share

In [ ]:
run("scripts/07_fig4_exposure_maps.py", "data/processed", f"{OUT}/Figure_4")
show(f"{OUT}/Figure_4.png")

## 08 · Table 3 — Moran's I and LISA high–high clusters
Moran's I values should match; permutation p-values use seed 2026.

In [ ]:
run("scripts/08_table3_moran_lisa.py", "data/processed", f"{OUT}/table3")
display(pd.read_csv(f"{OUT}/table3/Table3_moran_lisa.csv", encoding="utf-8-sig"))
same_csv(f"{OUT}/table3/Table3_moran_lisa.csv", "results/08-09_table3_spatial_clustering/Table3_moran_lisa.csv")

## 09 · Table 3 — space–time permutation scan (about 3 minutes)

In [ ]:
run("scripts/09_table3_spacetime_scan.py", "data/processed/hdc_saraburi_pm25_disease_tambon_monthly_2565_2569.csv",
    "data/processed/sat_pm25_tambon_summary.gpkg", f"{OUT}/table3")
display(pd.read_csv(f"{OUT}/table3/Fig5_scan_clusters.csv", encoding="utf-8-sig").head(12))
same_csv(f"{OUT}/table3/Fig5_scan_clusters.csv", "results/08-09_table3_spatial_clustering/Fig5_scan_clusters.csv")

## 10 · Fig. 5 — scan clusters (from step 09)

In [ ]:
run("scripts/10_fig5_scan_cluster_maps.py", f"{OUT}/table3/Fig5_scan_clusters.csv", "data/processed/sat_pm25_tambon_summary.gpkg", f"{OUT}/Figure_5")
show(f"{OUT}/Figure_5.png", 1100)

## 12 · Fig. 6 — within-tambon IRR by outcome and lag
Reads the stored NB-BYM2 results of notebook 11.

In [ ]:
run("scripts/12_fig6_irr_by_lag.py", "results/11_main_nb_bym2/Table5_IRR_long.csv", f"{OUT}/Figure_6")
show(f"{OUT}/Figure_6.png")

## 13 · Fig. 8 — predicted visit ratio, exceedance probability and surveillance priority

In [ ]:
run("scripts/13_fig8_predicted_risk_maps.py", "results/11_main_nb_bym2/Risk_all_tambons.csv", "data/processed/sat_pm25_tambon_summary.gpkg", f"{OUT}/Figure_8")
show(f"{OUT}/Figure_8.png", 1100)

## 15 · Extended panel, January 2022 – September 2026 (input of notebook 16 and step 17)

In [ ]:
run("scripts/15_build_extended_panel.py", "data/processed", f"{OUT}/analysis_panel_total_lag0_satscale.csv")
same_csv(f"{OUT}/analysis_panel_total_lag0_satscale.csv", "data/processed/analysis_panel_total_lag0_satscale.csv")

## 17 · Sensitivity (vii) — station-scale vs satellite-scale PM2.5 (Sect. 3.7)
Expected: satellite-scale IRR 1.003 (0.965–1.043); station-scale 0.968 (0.942–0.996).

In [ ]:
run("scripts/17_sensitivity_vii_station_scale.py", "data/processed")

## Summary
All outputs of this run are in `colab_outputs/`. Download them from the Files pane, or compare them with `figures/` and `results/`.

In [ ]:
for r, _, fs in sorted(os.walk(OUT)):
    for f in sorted(fs): print(os.path.join(r, f))